# FastAPI ML-сервис: быстрый прогон
Пошаговая проверка `/forward`, `/history`, `/stats`. Требуется запущенный Postgres/Chroma и зависимости.

## 0. Предусловия
1. Активировать venv.
2. Применить миграции: `alembic upgrade head`.
3. Запустить сервис: `uvicorn ml_service.main:app --host 0.0.0.0 --port 8000`.
4. Выполнить ячейки ниже.

In [12]:
import os, time, base64, json
import requests

BASE_URL = os.getenv('BASE_URL', 'http://localhost:8000')
JWT_SECRET = os.getenv('JWT_SECRET', 'devsecret') 

try:
    import jwt
    ADMIN_TOKEN = jwt.encode({'role': 'admin', 'iat': int(time.time())}, JWT_SECRET, algorithm='HS256')
except Exception as e:
    ADMIN_TOKEN = None
    print('Не удалось сгенерировать JWT автоматически:', e)

print('BASE_URL =', BASE_URL)
print('ADMIN_TOKEN есть' if ADMIN_TOKEN else 'ADMIN_TOKEN не сгенерирован')


BASE_URL = http://localhost:8000
ADMIN_TOKEN есть


## 1. Текстовый запрос к /forward
Используем ключ `text` (опционально `model`).

In [22]:
#Ollama
payload = {'text': 'Что такое фонд?', 'model': 'qwen2.5:7b-instruct'}
headers = {'Authorization': f"Bearer {ADMIN_TOKEN}"} if 'ADMIN_TOKEN' in globals() and ADMIN_TOKEN else {}
resp = requests.post(f"{BASE_URL}/forward", json=payload, headers=headers, timeout=60)
print('status', resp.status_code)
print(resp.text[:500])

status 200
{"answer":"Фонд занимается социальными инвестициями с стратегической целью поддержки развития системы общего образования и создания инклюзивной среды. Он финансирует свои собственные проекты и также поддерживает другие организации, соответствующие его миссии. Фонд состоит из 10 сотрудников с опытом управления социальными проектами в различных сферах."}


In [24]:
#Ollama
payload = {'text': 'Расскажи кратко, что такое облигация', 'model': 'qwen2.5:7b-instruct'}
headers = {'Authorization': f"Bearer {ADMIN_TOKEN}"} if 'ADMIN_TOKEN' in globals() and ADMIN_TOKEN else {}
resp = requests.post(f"{BASE_URL}/forward", json=payload, headers=headers, timeout=60)
print('status', resp.status_code)
print(resp.text[:500])

status 200
{"answer":"Облигация — это вид ценных бумаг, приобретение которых дает право на получение фиксированного дохода в виде погашения номинальной стоимости и процентов. Однако в контексте предоставленных материалов не указаны конкретные плюсы и минусы облигаций."}


## 2. multipart (демо с картинкой)


In [ ]:
png_bytes = base64.b64decode('iVBORw0KGgoAAAANSUhEUgAAAA4AAAAOCAYAAAAfSC3RAAAAHUlEQVR42mNgGAWjYBSMglEwCkb9D4YGhgYGBgAAAwCpkQo4h4Gv1gAAAABJRU5ErkJggg==')
files = {'image': ('sample.png', png_bytes, 'image/png')}
headers = {'X-Model': 'demo'}
resp = requests.post(f"{BASE_URL}/forward", files=files, headers=headers, timeout=60)
print('status', resp.status_code)
print(resp.text[:200])

status 200
{"image_base64":"iVBORw0KGgoAAAANSUhEUgAAAA4AAAAOCAYAAAAfSC3RAAAAEUlEQVR4nGNgGAWjYBQwYAUAAx4AASI5qeUAAAAASUVORK5CYII="}


## 3. История


In [26]:
resp = requests.get(f"{BASE_URL}/history", timeout=30)
print('status', resp.status_code)
print(resp.json() if resp.ok else resp.text)

status 200
[{'id': 4, 'created_at': '2025-12-26T08:54:51.583437', 'input_type': 'image', 'model': None, 'status': 'ok', 'error': None, 'duration_ms': 0.18429999909130856, 'text_len': None, 'token_count': None, 'image_width': 14, 'image_height': 14, 'response_preview': 'image_base64'}, {'id': 3, 'created_at': '2025-12-26T08:54:44.826751', 'input_type': 'text', 'model': 'qwen2.5:7b-instruct', 'status': 'ok', 'error': None, 'duration_ms': 2778.032499998517, 'text_len': 36, 'token_count': 5, 'image_width': None, 'image_height': None, 'response_preview': 'Облигация — это вид ценных бумаг, приобретение которых дает право на получение фиксированного дохода в виде погашения номинальной стоимости и процентов. Однако в контексте предоставленных материалов н'}, {'id': 2, 'created_at': '2025-12-26T08:54:32.720080', 'input_type': 'text', 'model': 'qwen2.5:7b-instruct', 'status': 'ok', 'error': None, 'duration_ms': 5639.9110999991535, 'text_len': 36, 'token_count': 5, 'image_width': None, 'image_hei

## 4. Статистика (admin JWT)


In [27]:
headers = {}
if ADMIN_TOKEN:
    headers['Authorization'] = f'Bearer {ADMIN_TOKEN}'
resp = requests.get(f"{BASE_URL}/stats", headers=headers, timeout=30)
print('status', resp.status_code)
print(resp.json() if resp.ok else resp.text)

status 200
{'count': 4, 'duration_ms': {'mean': 4310.075749999669, 'p50': 4208.971799998835, 'p95': 8344.8355000015, 'p99': 8726.707180001831}, 'text_len': {'mean': 29.0, 'p50': 36.0}, 'token_count': 4.333333333333333, 'image_width': {'mean': 14.0, 'p50': 14}, 'image_height': {'p50': 14}}


## 5. Очистка истории (admin JWT)


In [28]:
headers = {}
if ADMIN_TOKEN:
    headers['Authorization'] = f'Bearer {ADMIN_TOKEN}'
resp = requests.delete(f"{BASE_URL}/history", headers=headers, timeout=30)
print('status', resp.status_code)
print(resp.text)

status 200
{"deleted":4}
